# Modern Data Modeling
In production applications, we often need data structures that go beyond simple dictionaries. We need validation, type safety, and boilerplate-free classes. Two of the most popular tools for this are **Dataclasses** (built-in) and **Pydantic** (external library).

## Dataclasses
Introduced in Python 3.7, the `@dataclass` decorator automatically generates methods like `__init__`, `__repr__`, and `__eq__` based on type hints.

In [1]:
from dataclasses import dataclass, field

@dataclass(frozen=True) # frozen=True makes it immutable
class User:
    id: int
    username: str
    emails: list[str] = field(default_factory=list)

u1 = User(id=1, username="alex", emails=["alex@example.com"])
u2 = User(id=1, username="alex", emails=["alex@example.com"])

print(u1)
print(f"Are they equal? {u1 == u2}")

User(id=1, username='alex', emails=['alex@example.com'])
Are they equal? True


## `__post_init__` in Dataclasses
Use `__post_init__` to perform calculations or validation after the object has been initialized.

In [2]:
@dataclass
class Product:
    name: str
    unit_price: float
    quantity: int = 0
    total_cost: float = field(init=False) # Not passed to __init__

    def __post_init__(self):
        self.total_cost = self.unit_price * self.quantity

p = Product("Laptop", 1200.0, 3)
print(f"Total cost: {p.total_cost}")

Total cost: 3600.0


## Pydantic
While dataclasses are great, they don't perform runtime type validation. **Pydantic** is the industry standard for data validation and settings management (used heavily in FastAPI).

In [3]:
from pydantic import BaseModel, EmailStr, Field, ValidationError

class UserSchema(BaseModel):
    id: int
    username: str = Field(min_length=3, max_length=20)
    email: str # Use EmailStr if you have 'pydantic[email]' installed
    age: int = Field(gt=0, lt=120)

try:
    # Type conversion happens automatically (e.g., id="123" becomes 123)
    user = UserSchema(id="123", username="al", email="invalid-email", age=-5)
except ValidationError as e:
    print("Validation Errors found:")
    print(e.json(indent=2))

Validation Errors found:
[
  {
    "type": "string_too_short",
    "loc": [
      "username"
    ],
    "msg": "String should have at least 3 characters",
    "input": "al",
    "ctx": {
      "min_length": 3
    },
    "url": "https://errors.pydantic.dev/2.12/v/string_too_short"
  },
  {
    "type": "greater_than",
    "loc": [
      "age"
    ],
    "msg": "Input should be greater than 0",
    "input": -5,
    "ctx": {
      "gt": 0
    },
    "url": "https://errors.pydantic.dev/2.12/v/greater_than"
  }
]


## Summary
- **Dataclasses**: Best for internal data containers (low overhead, no external dependencies).
- **Pydantic**: Best for data arriving from external sources (APIs, Config files) where runtime validation is critical.
- **Pro-tip**: In Python 3.10+, you can also use `kw_only=True` in dataclasses to force keyword-only arguments!